# Workshop notebook 1: Why static mixture models are not enough

This notebook builds intuition for **ODE-constrained mixture modeling (ODE-MM)** using a simplified version of the conversion-process example from Hasenauer et al. (2014).

## Goals
- simulate snapshot data from **two hidden subpopulations**
- fit an **independent Gaussian mixture** at each time point
- see why component matching across time points is unstable
- motivate adding **mechanistic ODE constraints**

The biological/statistical motivation follows the paper:
> snapshot distributions can be described by mixtures, but independent fits across time points do not encode mechanistic links between conditions or time.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from sklearn.mixture import GaussianMixture

# Create a random number generator with a fixed seed
rng = np.random.default_rng(4)

# Parameters for visualization
plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

## 1. Define a simple mechanistic process

We use a very simple ODE to describe the dynamics of means of two subpopulations:

$\frac{dB}{dt} = k(B_\infty - B)$

Interpretation:
- $k$ controls how quickly a subpopulation responds
- both subpopulations follow the *same model form*
- they differ only in the response rate $k$

This mirrors the logic of the paper's conversion-process example: one hidden subpopulation is more responsive than the other.

#### Plug in the solution of the ODE into the *mean-response* function

In [ ]:
def mean_response(t, k, B0=0.22, B_inf=0.48):
    return

#### Visualize the dynamics of two subpopulation means for different response rates *k = 0.45* & *k = 2.2*

In [ ]:
times = np.linspace(0,1,1001)

subpop1 = mean_response()
subpop2 = mean_response()

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(times, subpop1, "-", label="pop 1")
ax.plot(times, subpop2, "-", label="pop 2")
ax.set_ylabel("concentration B")
ax.set_xlabel("time")
ax.legend()

plt.title("Mean concentration for two hidden subpopulations", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ground-truth hidden subpopulations parameters
true = {
    "weights": np.array([0.7, 0.3]),
    "k": np.array([0.45, 2.2]),
    "sigma": np.array([0.018, 0.022]),
}

# measurements are done at times
times = np.array([0.0, 0.1, 0.2, 0.3, 0.5, 1.0])

In [ ]:
def sample_snapshot(t, n_cells=500):
    
    weights = true["weights"]
    k = true["k"]
    sigma = true["sigma"]

    # Assign each simulated cell to a subpopulation, for example, for n_cells=7:
    # [0 1 0 0 1 1 1]
    # [0 0 0 1 0 0 0]
    # [1 0 0 0 0 0 1] etc
    z = rng.choice(2, size=n_cells, p=weights)
    means = np.array([mean_response(t, kk) for kk in k])
    y = rng.normal(means[z], sigma[z], size=n_cells)
    
    return np.clip(y, 0, 1), z, means

# Generate measured snapshot distributions at t=times
data = {}
for t in times:
    y, z, means = sample_snapshot(t, n_cells=700)
    data[t] = {"y": y, "z": z, "means": means}

#### Visualize the snapshop data

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 6), sharex=True, sharey=True)
axes = axes.ravel()

xs = np.linspace(0.15, 0.55, 400)

for ax, t in zip(axes, times):
    y = data[t]["y"]
    means = data[t]["means"]
    ax.hist(y, bins=35, density=True, alpha=0.45, label="simulated cells")
    ax.set_title(f"t = {t:g}")
    ax.set_xlabel("measured B")
    ax.set_ylabel("density")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.04))
fig.suptitle("Ground-truth snapshot distributions", y=1.08, fontsize=14)
fig.tight_layout()
plt.show()

### Small exercise

Try one or more of these changes and re-generate the snapshot data:
1. increase the overlap by making the two `k` values closer
2. increase the noise (`sigma`)
3. reduce `n_cells`

### Question:
- when does it become hard to tell if there are one or two subpopulations?

### The analysis of population snapshop data can be approached using a multitude of statistical methods: 
- thresholding
- density-based methods
- mixture modeling

The result of such analysis should look like it is shown on the following plot

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 6), sharex=True, sharey=True)
axes = axes.ravel()

xs = np.linspace(0.15, 0.55, 400)

for ax, t in zip(axes, times):
    y = data[t]["y"]
    means = data[t]["means"]
    ax.hist(y, bins=35, density=True, alpha=0.45, label="simulated cells")
    for i, (w, m, s) in enumerate(zip(true["weights"], means, true["sigma"])):
        ax.plot(xs, w * norm.pdf(xs, m, s), lw=2, label=f"subpop {i+1}" if t == times[0] else None)
    ax.set_title(f"t = {t:g}")
    ax.set_xlabel("measured B")
    ax.set_ylabel("density")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.04))
fig.suptitle("Ground-truth snapshot distributions", y=1.08, fontsize=14)
fig.tight_layout()
plt.show()

## 2. Fit a Gaussian mixture **independently** at each time point

This is exactly the setup that creates the matching problem:
- each snapshot is fit well
- but component labels can switch
- and independent fits do not enforce smooth mechanistic trajectories

In [ ]:
def fit_gmm_per_timepoint(data):
    fits = {}
    for t, d in data.items():
        y = d["y"].reshape(-1, 1)
        gmm = GaussianMixture(n_components=2, covariance_type="full", random_state=0, n_init=10)
        gmm.fit(y)

        means = gmm.means_.ravel()
        stds = np.sqrt(gmm.covariances_.reshape(-1))
        weights = gmm.weights_.ravel()

        fits[t] = {
            "gmm": gmm,
            "means": means,
            "stds": stds,
            "weights": weights
        }
    return fits

gmm_fits = fit_gmm_per_timepoint(data)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 6), sharex=True, sharey=True)
axes = axes.ravel()
xs = np.linspace(0.15, 0.55, 400)

for ax, t in zip(axes, times):
    y = data[t]["y"]
    fit = gmm_fits[t]
    ax.hist(y, bins=35, density=True, alpha=0.4, color="0.75", label="data")
    total = np.zeros_like(xs)
    for i, (w, m, s) in enumerate(zip(fit["weights"], fit["means"], fit["stds"])):
        comp = w * norm.pdf(xs, m, s)
        total += comp
        ax.plot(xs, comp, lw=2, label=f"GMM comp {i+1}" if t == times[0] else None)
    ax.plot(xs, total, color="k", lw=2, ls="--", label="mixture fit" if t == times[0] else None)
    ax.set_title(f"t = {t:g}")
    ax.set_xlabel("measured B")
    ax.set_ylabel("density")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, ncol=4, loc="upper center", bbox_to_anchor=(0.5, 1.05))
fig.suptitle("Independent Gaussian mixture fits", y=1.1, fontsize=14)
fig.tight_layout()
plt.show()

### Questions:
- is the fitting quality satisfying?
- are there any incosistencies that you can spot?

## 3. Track fitted component means across time

A key issue appears here:
the component numbering from a standard GMM is arbitrary.

So even when each time point is fit well, the implied trajectories can look inconsistent unless you manually match components.

In [ ]:
raw_means = np.array([gmm_fits[t]["means"] for t in times])
sorted_means = np.sort(raw_means, axis=1)

true_means = np.array([[mean_response(t, kk) for kk in true["k"]] for t in times])
true_sorted = np.sort(true_means, axis=1)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(times, raw_means[:, 0], "o-", label="raw comp 1")
ax.plot(times, raw_means[:, 1], "o-", label="raw comp 2")
ax.plot(times, sorted_means[:, 0], "s--", label="sorted comp low")
ax.plot(times, sorted_means[:, 1], "s--", label="sorted comp high")
ax.plot(times, true_sorted[:, 0], lw=3, alpha=0.6, label="true low")
ax.plot(times, true_sorted[:, 1], lw=3, alpha=0.6, label="true high")
ax.set_xlabel("time")
ax.set_ylabel("component mean")
ax.set_title("Independent fits require post hoc component matching")
ax.legend()
plt.show()

### Small exercise

Try one or more of these changes and rerun the notebook:
1. increase the overlap by making the two `k` values closer
2. increase the noise (`sigma`)
3. reduce `n_cells`

### Questions:
- when does the independent GMM become unreliable?
- when do component labels switch?
- when does it become hard to tell if there are one or two subpopulations?

This is the point where ODE constraints become useful.

## 4. Take-home message

A static mixture model explains each histogram separately.

An **ODE-constrained mixture model** instead says:
- each component corresponds to a biological subpopulation
- each subpopulation evolves according to a mechanistic ODE
- all time points are fit **simultaneously**

Move on to notebook 2 to fit a simple ODE-constrained mixture model.